# Agent Lab: The Self-Correction Loop
This notebook demonstrates how an **Auditor Agent** can automatically detect and fix mistakes in model output without human intervention.

### The Agentic Flow:
1. **Validation:** A Python function checks for errors (e.g., math mismatch).
2. **Trigger:** If an error is found, the Agent is called.
3. **Self-Correction:** The Agent receives the error and the context, then provides a fixed JSON.

In [ ]:
import json
from openai import OpenAI

llm_client = OpenAI(base_url="http://localhost:8001/v1", api_key="EMPTY")
LLM_MODEL = "Qwen/Qwen3-4B-AWQ"

def validate_math(data):
    items = data.get("line_items", [])
    calculated_total = sum(item.get("line_total", 0) for item in items)
    extracted_total = data.get("financials", {}).get("total_amount", 0)
    
    if abs(calculated_total - extracted_total) > 0.01:
        return False, f"Math Error: Items sum to {calculated_total} but total says {extracted_total}"
    return True, "Math is correct."

## 1. Simulate a Broken Extraction
Here, we mock a JSON where the math doesn't add up ($10 + $20 != $40).

In [ ]:
mock_broken_json = {
    "line_items": [
        {"description": "Pen", "line_total": 10.0},
        {"description": "Paper", "line_total": 20.0}
    ],
    "financials": {
        "total_amount": 40.0
    }
}

mock_ocr_text = "Office Supplies: Pen - $10, Paper - $20. Total Due: $30 (Wait, typo in OCR: $40)"

is_valid, error_msg = validate_math(mock_broken_json)
print(f"Validation Result: {is_valid}")
print(f"Error Found: {error_msg}")

## 2. The Agentic Correction Call
The Agent sees the error and the original text to decide the fix.

In [ ]:
if not is_valid:
    print("Triggering Auditor Agent...")
    
    AUDITOR_PROMPT = f"""You are a detailed Financial Auditor.
    A previous AI extracted this JSON: {json.dumps(mock_broken_json)}
    
    It FAILED validation with this error: {error_msg}
    
    Look at the original OCR text below and return a CORRECTED JSON.
    Original Text: {mock_ocr_text}
    """
    
    response = llm_client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "system", "content": AUDITOR_PROMPT}],
        temperature=0.0
    )
    
    fixed_json = response.choices[0].message.content
    print("\n--- AGENT FIXED OUTPUT ---")
    print(fixed_json)